# Defects on one lattice: arrangement against a pair expansion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/21_defects_on_a_lattice.ipynb)

A dopant study asks the same question many times: what does this arrangement of defects cost,
on this one host lattice? The host never changes. Only a few sites do. This notebook uses the
2DMD set (Huang et al., npj 2D Mater. Appl. 7, 6 (2023)): 5,933 cells of MoS2 and 5,933 of WSe2,
each an 8 x 8 supercell of 192 sites with one to three point defects (vacancies and
substitutions), and the DFT formation energy of each cell.

Two ideas make the field kernel work here. The **difference field**: every cell is the host plus
a change on one to three sites, and the overlap kernel is bilinear, so a model can work on the
changes alone. The **host symmetry**: two arrangements related by an operation of the host have
the same energy, so the kernel is averaged over the 768 operations of the supercell. The linear
average of the kernel only counts the defects of each kind. Averaging the Gaussian layer instead
keeps their arrangement. The notebook asks three questions.

1. What do the counts of defect kinds, and the arrangement kernel, give?
2. What does a **pair expansion** give: one energy for each kind of defect and one for each pair
   of kinds at each neighbour distance, fitted by linear regression? This is the standard cheap
   model for defects on a lattice, and the reference that is new in this notebook.
3. Trained on the cells with one and two defects, which model predicts the cells with three?

The first run downloads the data (5 MB) into `benchmarks/cache/2dmd`. The fits on all cells are
read from the stored results of `benchmarks/run_2dmd.py`.

Running time: about one minute, after the download.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## The series

`prepare` in the benchmark script reads the cells (downloading them the first time) and reduces
each one to its difference from the host: the sites it leaves empty and the sites that carry
another element. Every cell is labelled with its class under the host symmetry, and no two cells
of a material share a class.

In [2]:
import contextlib
import io
import json
import os
import sys

os.environ["CMF_KERNEL_CACHE"] = "0"
sys.path.insert(0, "benchmarks")

import matplotlib.pyplot as plt
import numpy as np

import run_2dmd as D                      # the data, kernels, folds and fits of item M14

prep = D.prepare("MoS2")
recs = prep["records"]
y_all = np.array([r[D.TARGET] for r in recs])
nd_all = np.array([r["n_defects"] for r in recs])
kinds = {}
for r in recs:
    for _, c in D.defect_changes(prep, r):
        kinds[c] = kinds.get(c, 0) + 1
print("MoS2: %d cells on a host of %d sites (%s); %d classes under the host symmetry" % (
    len(recs), len(prep["host_symbols"]),
    ", ".join("%d %s" % (prep["host_symbols"].count(s), s) for s in sorted(set(prep["host_symbols"]))), prep["n_classes"]))
print("cells by number of defects:", {int(k): int(v) for k, v in zip(*np.unique(nd_all, return_counts=True))})
print("defects by kind:", kinds, "('-S' is a sulfur vacancy, '+Se' a selenium on a sulfur site)")
print("formation energy of a cell: %.2f to %.2f eV, mean %.2f" % (y_all.min(), y_all.max(), y_all.mean()))

MoS2: 5933 cells on a host of 192 sites (64 Mo, 128 S); 5933 classes under the host symmetry
cells by number of defects: {1: 4, 2: 127, 3: 5802}
defects by kind: {'-S': 5900, '+Se': 5900, '-Mo': 2932, '+W': 2932} ('-S' is a sulfur vacancy, '+Se' a selenium on a sulfur site)
formation energy of a cell: 0.17 to 12.62 eV, mean 6.47


## Kernels and a pair expansion, fitted here

The cell below takes 600 cells of MoS2: all 131 with one or two defects, and 469 with three
(200 cells if Numba, which compiles the symmetry average, is not installed). It
builds two kernels on the difference fields averaged over the 768 operations of the host: the
**counts** kernel (the linear average, which adds one energy per kind of defect) and the
**arrangement** kernel (the averaged Gaussian layer). Each is fitted by the benchmark's nested
cross-validation, five random folds with the setting and the ridge chosen inside each training
part, and once more trained on the cells with one and two defects and tested on the cells with
three. The error is given per defect site, the error of the cell divided by its number of
defects, which is what the literature on this set reports.

In [3]:
D.ALPHAS, D.GAMMAS = (1.0,), (0.25,)       # one attenuation and one layer width, to keep this cell short
try:
    import numba                           # the compiled symmetry average; without it 200 cells, not 600
    n_live, engine = 600, "numba"
except ImportError:
    n_live, engine = 200, "numpy"
sub = D.pick(prep, n_live)
series = D.Series(prep, sub)
y, nd = series.y, series.n_defects.astype(float)
few = nd <= 2
folds = D.folds(len(y))
extra = [(np.where(few)[0], np.where(~few)[0])]
live = {}
for kind in ("counts", "arrangement"):
    with contextlib.redirect_stdout(io.StringIO()):          # the benchmark logs every fold
        Ks = D.unit_diagonal(D.kernels_for(kind, series, "notebook", engine=engine))
        live[kind] = (D.evaluate(Ks, y, series.n_defects, series.classes, folds),
                      D.evaluate(Ks, y, series.n_defects, series.classes, extra))
print("%d cells of MoS2, %d of them with three defects" % (len(y), int((~few).sum())))
print("%-14s %22s %32s" % ("kernel", "random folds, meV/site", "one and two defects -> three"))
for kind, (rnd, ext) in live.items():
    print("%-14s %22.1f %32.1f" % (kind, 1000 * rnd["mae_site"], 1000 * ext["mae_site"]))

600 cells of MoS2, 469 of them with three defects
kernel         random folds, meV/site     one and two defects -> three
counts                           40.2                             42.0
arrangement                       1.9                              4.3


The pair expansion needs no kernel. Its columns are counts: the defects of each kind, and the
pairs of each two kinds in each shell of host-site distances up to a cut-off. `pair_features` and
`pair_evaluate` of the benchmark build and fit it, with the ridge chosen inside each fold. It has
two forms. One has a fitted constant. The other fixes the constant at zero, because the formation
energy of a cell without defects is zero by definition. The next cell fits both, with pairs up
to 12 Å, on the same 600 cells and folds.

In [4]:
X, keys, shells = D.pair_features(prep, sub, 12.0)
print("pairs up to 12 A: %d shells of distance, %d columns, for example %s" % (len(shells), len(keys), keys[:3]))
print("%-30s %22s %32s" % ("pair expansion", "random folds, meV/site", "one and two defects -> three"))
for form in D.PAIR_FORMS:
    rnd = D.pair_evaluate(X, y, nd, folds, form)
    ext = D.pair_evaluate(X, y, nd, extra, form)
    print("%-30s %22.1f %32.1f" % (form, 1000 * rnd["mae_site"], 1000 * ext["mae_site"]))

pairs up to 12 A: 25 shells of distance, 96 columns, for example ['n +Se', 'n +W', 'n -Mo']
pair expansion                 random folds, meV/site     one and two defects -> three


with a constant                                   2.1                            252.5


constant zero                                     2.1                              7.9


On these 600 cells the counts kernel errs by 40 meV per site and the arrangement kernel by 1.9: the
arrangement of the defects carries almost all of the energy that the kinds alone leave. The pair
expansion with pairs up to 12 Å reaches 2.1 meV per site with random folds in both forms, so it is
nearly as good as the arrangement kernel.

The two forms part when the model is trained on the cells with one and two defects and tested on
the cells with three. With a fitted constant the pair expansion fails (252 meV per site). In a cell
with one or two defects the number of pairs is always the number of defects minus one, so the pair
terms and the constant can trade a constant between them, and the cells with three defects, which
have three pairs, pay for that trade. With the constant at zero there is nothing to trade, and the
pair expansion gives 7.9 meV per site against 4.3 for the arrangement kernel and 42 for the counts.
The first form was written first; the second was added after the failure was seen, for the reason
just given, and both are reported.

## All cells, from the stored runs

`run_2dmd.py` fits the four kernels of item M14 on 2,000 and on all 5,933 cells of each material,
and `run_2dmd.py --stage pairs` fits the pair expansion at four cut-offs (4, 6, 8 and 12 Å) and in
both forms on the same cells and folds. The table reads both stored results for all 5,933 cells.

In [5]:
pairs = json.load(open(os.path.join(D.RESULTS, "pairs.json")))
for material in ("MoS2", "WSe2"):
    run = pairs["runs"]["%s | 5933" % material]
    s = run["stored"]
    print("%s, all 5933 cells: MAE per site in meV" % material)
    print("   %-38s %8s %22s" % ("model", "random", "one and two -> three"))
    for k in D.KERNELS:
        print("   %-38s %8.1f %22.1f" % (k + " kernel", 1000 * s[k]["mae_site"], 1000 * s[k]["extrapolation_mae_site"]))
    for c, rec in run["cutoffs"].items():
        print("   %-38s %8.1f %22.1f" % ("pairs up to %g A, %s" % (rec["cutoff"], rec["form"]),
                                        1000 * rec["random"]["mae_site"], 1000 * rec["extrapolation"]["mae_site"]))
    best = run["cutoffs"]["constant zero | 12"]["paired"]
    for design in ("random", "extrapolation"):
        iv = best["%s: pairs - arrangement" % design]
        print("   pairs up to 12 A, constant zero, minus the arrangement kernel, %s: %+.1f meV [%+.1f, %+.1f]" % (
            design, iv["diff"], iv["low"], iv["high"]))
    print()

MoS2, all 5933 cells: MAE per site in meV
   model                                    random   one and two -> three
   fixed kernel                               41.9                  613.9
   counts kernel                              41.0                   41.2
   arrangement kernel                          1.9                    4.7
   autocorrelation kernel                      3.0                 1292.7
   pairs up to 4 A, with a constant           14.7                  274.8
   pairs up to 6 A, with a constant            3.4                  267.9
   pairs up to 8 A, with a constant            2.9                  262.5
   pairs up to 12 A, with a constant           2.2                  248.4
   pairs up to 4 A, constant zero             14.7                  143.8
   pairs up to 6 A, constant zero              3.4                   25.1
   pairs up to 8 A, constant zero              2.8                   25.1
   pairs up to 12 A, constant zero             2.2                    

Over all 5,933 cells the picture of the live fit holds, and the second material changes the end.

- **Random folds.** With pairs up to 12 Å the pair expansion is behind the arrangement kernel by
  0.2 meV per site on MoS2 and 0.3 on WSe2, and both intervals exclude zero. It is ahead of the
  periodic autocorrelation (2.2 against 3.0, and 4.7 against 6.0). With pairs up to 4 Å it errs by
  15 and 19 meV: the neighbours within 6 to 12 Å matter.
- **From one and two defects to three.** With the constant at zero and pairs up to 12 Å, the pair
  expansion is behind the arrangement kernel on MoS2 (8.2 against 4.7, +3.5 [+3.5, +3.6]) and far
  ahead of it on WSe2 (14.9 against 37.7, -22.8 [-23.7, -21.9]). Shorter cut-offs extrapolate
  worse (25 to 144 meV), except 8 Å on WSe2.

So on random folds the arrangement kernel reads little more than which pairs of defects are
neighbours, and which of the two extrapolates better depends on the material and on two free
choices of the reference, its constant and its cut-off.

## What each kind of defect costs

The counts model is linear in the difference fields, so its prediction splits into one number
for each defect (decomposition rule 3). `run_2dmd.py --stage reading` stores them for 400 cells.

In [6]:
reading = json.load(open(os.path.join(D.RESULTS, "reading.json")))
for material in ("MoS2", "WSe2"):
    r = reading[material]
    print("%s, %d cells, training R2 %+.3f:" % (material, r["n"], r["r2_train"]))
    for name, k in sorted(r["kinds"].items()):
        print("   %-4s in %4d cells: %+.3f eV each (spread over the cells %.1e)" % (name, k["n"], k["mean"], k["sd"]))

MoS2, 400 cells, training R2 +0.994:
   +Se  in  356 cells: -2.280 eV each (spread over the cells 3.7e-12)
   +W   in  160 cells: -6.785 eV each (spread over the cells 3.9e-11)
   -Mo  in  331 cells: +6.978 eV each (spread over the cells 5.5e-11)
   -S   in  734 cells: +2.594 eV each (spread over the cells 8.4e-12)
WSe2, 400 cells, training R2 +0.995:
   +Mo  in  160 cells: -5.506 eV each (spread over the cells 5.0e-11)
   +S   in  379 cells: -3.027 eV each (spread over the cells 1.8e-10)
   -Se  in  734 cells: +2.619 eV each (spread over the cells 1.6e-10)
   -W   in  331 cells: +5.091 eV each (spread over the cells 3.3e-11)


Every kind of defect gets one number, the same in every cell to about ten decimals, as the symmetry
requires. In this reading a substitution is the host atom removed plus the new atom added, so
selenium for sulfur in MoS2 costs 2.594 - 2.280 = 0.314 eV, and a sulfur vacancy 2.594 eV. The
single-defect cells of the set give 0.28 and 2.65 eV. This is what the counts kernel knows, and it
is worth 41 meV per site. The arrangement kernel and the pair expansion add the interaction of the
defects that sit near each other.

## What this shows and what it does not

**What it shows.**

- On one host lattice the symmetry-averaged kernel of the difference fields predicts the energy of
  a defect arrangement to 1.9 meV per site (MoS2) and 4.4 (WSe2), against 41 for one energy per
  kind of defect.
- A pair expansion, one energy per kind and one per pair of kinds at each distance up to 12 Å,
  fitted by linear regression, comes within 0.2 to 0.3 meV per site of it. That is the reference a
  defect model on a lattice has to be compared with, and the arrangement kernel is only just ahead.
- A reference has to be fitted in its right form. With a fitted constant the pair expansion cannot
  extrapolate from two defects to three; with the constant at zero, which is what a formation energy
  is, it can, and on WSe2 it extrapolates better than the kernel.

**What it does not show.**

- That one of the two models is better in general. The answer to the extrapolation depends on the
  material and on the cut-off of the pairs.
- Anything about relaxation. The cells are the ideal, unrelaxed supercell with its defects, so
  whatever the relaxation does is inside the label and not in the description.
- How the models compare with the published ones on the authors' own split (2.6 and 4.8 meV per
  site for a sparse graph network). These folds are not that split.